Purpose: Run enrichment (Fisher's exact test) for each Pathway (and maybe also gene family) in polynomial modeling results.<br>
Author: Anna Pardo<br>
Date initiated: July 28, 2026

In [1]:
# import modules
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os
import statistics
import scipy.stats as stats
import numpy as np
from statsmodels.stats.multitest import fdrcorrection
from venn import venn
from matplotlib.patches import Patch
import json

In [2]:
# enrichment approach (Fisher's exact test): are genes in a given pathway and/or gene family more likely to be significant
## in each Factor?
# table: isPathway/isFamily, isSignificant_for_Factor

# start with each pathway (including Other)

# load modeling results
mres_annot = pd.read_csv("./polymod3_fullmodel_results_withannot.txt",sep="\t",header="infer")
mres_annot.head()

,Sum Sq,Df,F value,Pr(>F),Factors,GeneID,FDR_p,Pathway,gene_family,subgenome,gene_name_unique,syntelogID,Description
0,0.465785,1,1.333161,2.485871e-01,(Intercept),Yucal.11G006100.v2.1,3.590887e-01,Clock,TOC1,Ya,Ya_TOC1_1,recip_syn19224,Two-component response regulator-like PRR1
1,147.216687,3,140.453660,3.730225e-73,"poly(ZT, 3)",Yucal.11G006100.v2.1,3.503564e-71,Clock,TOC1,Ya,Ya_TOC1_1,recip_syn19224,Two-component response regulator-like PRR1
2,0.757162,1,2.167136,1.413804e-01,treat,Yucal.11G006100.v2.1,2.276692e-01,Clock,TOC1,Ya,Ya_TOC1_1,recip_syn19224,Two-component response regulator-like PRR1
3,2.328701,2,3.332583,3.619521e-02,phys,Yucal.11G006100.v2.1,7.202747e-02,Clock,TOC1,Ya,Ya_TOC1_1,recip_syn19224,Two-component response regulator-like PRR1
4,0.015806,3,0.015080,9.974731e-01,"poly(ZT, 3):treat",Yucal.11G006100.v2.1,9.992700e-01,Clock,TOC1,Ya,Ya_TOC1_1,recip_syn19224,Two-component response regulator-like PRR1


In [6]:
allgenes = pd.DataFrame(mres_annot["GeneID"].unique())
allgenes.rename(columns={0:"GeneID"},inplace=True)

In [7]:
# function to make isPathway & isSig columns for a given pathway & factor
def data_setup(path,fac,col="Pathway",res=mres_annot,gid=allgenes):
    pathdf = res[res[col]==path]
    facdf = res[res["Factors"]==fac]
    
    pathornot = []
    sigornot = []
    for g in list(gid["GeneID"]):
        if g in list(pathdf["GeneID"]):
            pathornot.append("Yes")
        else:
            pathornot.append("No")
        if g in list(facdf[facdf["FDR_p"]<0.05]["GeneID"]):
            sigornot.append("Yes")
        else:
            sigornot.append("No")
    gid["is"+path] = pathornot
    gid["isSig_"+fac] = sigornot
    return gid

In [16]:
# define a function to run the whole Fisher's test
def run_fisher(path,fac,col="Pathway",res=mres_annot,gid=allgenes):
    df = data_setup(path,fac,col,res,gid)
    # set up the table for Fisher's exact test
    data = pd.crosstab(index=df["is"+path],columns=df["isSig_"+fac])
    # run Fisher's exact test
    odds_ratio, p_value = stats.fisher_exact(data)
    print('odds ratio: ' + str(odds_ratio))
    print('p-value: ' + str(p_value))
    return [odds_ratio, p_value]


In [12]:
mres_annot["Factors"].unique()

array(['poly(ZT, 3)', 'treat', 'phys', 'poly(ZT, 3):treat',
       'poly(ZT, 3):phys', 'treat:phys', 'poly(ZT, 3):treat:phys'],
      dtype=object)

In [11]:
# drop Residuals & Intercept from mres_annot
mres_annot = mres_annot[~mres_annot["Factors"].isin(["(Intercept)","Residuals"])]

In [18]:
# loop through pathway-factor combinations & run enrichment tests
dfdict = {"Pathway":[],"Factor":[],"p-value":[],"odds_ratio":[]}
for i in mres_annot["Pathway"].unique():
    for j in mres_annot["Factors"].unique():
        print("Pathway: "+i+", Factor: "+j)
        modresult = run_fisher(i,j)
        dfdict["p-value"].append(modresult[1])
        dfdict["odds_ratio"].append(modresult[0])
        dfdict["Pathway"].append(i)
        dfdict["Factor"].append(j)

Pathway: Clock, Factor: poly(ZT, 3)
odds ratio: inf
p-value: 0.4244332697489238
Pathway: Clock, Factor: treat
odds ratio: 0.5380492610837438
p-value: 0.026994840944103593
Pathway: Clock, Factor: phys
odds ratio: 0.6081893212959748
p-value: 0.09072098216570515
Pathway: Clock, Factor: poly(ZT, 3):treat
odds ratio: 0.39243763551023325
p-value: 0.0019853198494970935
Pathway: Clock, Factor: poly(ZT, 3):phys
odds ratio: 1.42031962861183
p-value: 0.24394077694471694
Pathway: Clock, Factor: treat:phys
odds ratio: 0.33968241551939926
p-value: 0.013497104464235334
Pathway: Clock, Factor: poly(ZT, 3):treat:phys
odds ratio: 0.6387908546877398
p-value: 0.621929705811697
Pathway: CAM, Factor: poly(ZT, 3)
odds ratio: 0.06207926783148022
p-value: 3.3327462429428513e-18
Pathway: CAM, Factor: treat
odds ratio: 0.6756307888853401
p-value: 0.1046513371096941
Pathway: CAM, Factor: phys
odds ratio: 1.0455992851119529
p-value: 0.9055965477781962
Pathway: CAM, Factor: poly(ZT, 3):treat
odds ratio: 0.250500648

In [19]:
enrichres = pd.DataFrame(dfdict)

In [20]:
enrichres.head()

,Pathway,Factor,p-value,odds_ratio
0,Clock,"poly(ZT, 3)",0.424433,inf
1,Clock,treat,0.026995,0.538049
2,Clock,phys,0.090721,0.608189
3,Clock,"poly(ZT, 3):treat",0.001985,0.392438
4,Clock,"poly(ZT, 3):phys",0.243941,1.420320


In [21]:
# run FDR
enrichres["FDR_p"] = fdrcorrection(enrichres["p-value"])[1]
enrichres.head()

,Pathway,Factor,p-value,odds_ratio,FDR_p
0,Clock,"poly(ZT, 3)",0.424433,inf,0.540188
1,Clock,treat,0.026995,0.538049,0.053990
2,Clock,phys,0.090721,0.608189,0.146549
3,Clock,"poly(ZT, 3):treat",0.001985,0.392438,0.005559
4,Clock,"poly(ZT, 3):phys",0.243941,1.420320,0.341517


In [22]:
enrichres[enrichres["FDR_p"]<0.05]

,Pathway,Factor,p-value,odds_ratio,FDR_p
3,Clock,"poly(ZT, 3):treat",1.985320e-03,0.392438,5.558896e-03
5,Clock,treat:phys,1.349710e-02,0.339682,3.334579e-02
7,CAM,"poly(ZT, 3)",3.332746e-18,0.062079,4.665845e-17
10,CAM,"poly(ZT, 3):treat",3.382959e-07,0.250501,1.578714e-06
11,CAM,"poly(ZT, 3):phys",1.593892e-02,0.424857,3.347173e-02
14,PhotResp,"poly(ZT, 3)",2.594387e-07,0.121336,1.362053e-06
18,PhotResp,"poly(ZT, 3):phys",9.534431e-07,0.000000,4.004461e-06
21,N-metab,"poly(ZT, 3)",1.571541e-11,0.078362,1.650118e-10
24,N-metab,"poly(ZT, 3):treat",6.180753e-04,0.363259,2.163264e-03
25,N-metab,"poly(ZT, 3):phys",1.436262e-02,0.345650,3.347173e-02


In [23]:
# save results
enrichres.to_csv("./pathways_enrichmentres_bymodelfactor.csv",sep=",",header=True,index=False)
enrichres[enrichres["FDR_p"]<0.05].to_csv("./sigpathways_enrichmentres_bymodelfactor.csv",sep=",",header=True,index=False)

In [24]:
# actually, split by parental origin
dfdict = {"Pathway":[],"Parental Origin":[],"Factor":[],"p-value":[],"odds_ratio":[]}
for s in ["Ya","Yf"]:
    df = mres_annot[mres_annot["subgenome"]==s]
    for i in df["Pathway"].unique():
        for j in df["Factors"].unique():
            print("Pathway: "+i+", Factor: "+j)
            modresult = run_fisher(i,j)
            dfdict["p-value"].append(modresult[1])
            dfdict["odds_ratio"].append(modresult[0])
            dfdict["Pathway"].append(i)
            dfdict["Factor"].append(j)
            dfdict["Parental Origin"].append(s)

Pathway: Clock, Factor: poly(ZT, 3)
odds ratio: inf
p-value: 0.4244332697489238
Pathway: Clock, Factor: treat
odds ratio: 0.5380492610837438
p-value: 0.026994840944103593
Pathway: Clock, Factor: phys
odds ratio: 0.6081893212959748
p-value: 0.09072098216570515
Pathway: Clock, Factor: poly(ZT, 3):treat
odds ratio: 0.39243763551023325
p-value: 0.0019853198494970935
Pathway: Clock, Factor: poly(ZT, 3):phys
odds ratio: 1.42031962861183
p-value: 0.24394077694471694
Pathway: Clock, Factor: treat:phys
odds ratio: 0.33968241551939926
p-value: 0.013497104464235334
Pathway: Clock, Factor: poly(ZT, 3):treat:phys
odds ratio: 0.6387908546877398
p-value: 0.621929705811697
Pathway: CAM, Factor: poly(ZT, 3)
odds ratio: 0.06207926783148022
p-value: 3.3327462429428513e-18
Pathway: CAM, Factor: treat
odds ratio: 0.6756307888853401
p-value: 0.1046513371096941
Pathway: CAM, Factor: phys
odds ratio: 1.0455992851119529
p-value: 0.9055965477781962
Pathway: CAM, Factor: poly(ZT, 3):treat
odds ratio: 0.250500648

In [25]:
sgres = pd.DataFrame(dfdict)
sgres.head()

,Pathway,Parental Origin,Factor,p-value,odds_ratio
0,Clock,Ya,"poly(ZT, 3)",0.424433,inf
1,Clock,Ya,treat,0.026995,0.538049
2,Clock,Ya,phys,0.090721,0.608189
3,Clock,Ya,"poly(ZT, 3):treat",0.001985,0.392438
4,Clock,Ya,"poly(ZT, 3):phys",0.243941,1.420320


In [26]:
sgres["FDR_p"] = fdrcorrection(sgres["p-value"])[1]
sgres.head()

,Pathway,Parental Origin,Factor,p-value,odds_ratio,FDR_p
0,Clock,Ya,"poly(ZT, 3)",0.424433,inf,0.540188
1,Clock,Ya,treat,0.026995,0.538049,0.053990
2,Clock,Ya,phys,0.090721,0.608189,0.146549
3,Clock,Ya,"poly(ZT, 3):treat",0.001985,0.392438,0.005559
4,Clock,Ya,"poly(ZT, 3):phys",0.243941,1.420320,0.341517


In [27]:
sigsg = sgres[sgres["FDR_p"]<0.05]
sigsg

,Pathway,Parental Origin,Factor,p-value,odds_ratio,FDR_p
3,Clock,Ya,"poly(ZT, 3):treat",1.985320e-03,0.392438,5.558896e-03
5,Clock,Ya,treat:phys,1.349710e-02,0.339682,3.334579e-02
7,CAM,Ya,"poly(ZT, 3)",3.332746e-18,0.062079,4.665845e-17
10,CAM,Ya,"poly(ZT, 3):treat",3.382959e-07,0.250501,1.578714e-06
11,CAM,Ya,"poly(ZT, 3):phys",1.593892e-02,0.424857,3.347173e-02
14,PhotResp,Ya,"poly(ZT, 3)",2.594387e-07,0.121336,1.362053e-06
18,PhotResp,Ya,"poly(ZT, 3):phys",9.534431e-07,0.000000,4.004461e-06
21,N-metab,Ya,"poly(ZT, 3)",1.571541e-11,0.078362,1.650118e-10
24,N-metab,Ya,"poly(ZT, 3):treat",6.180753e-04,0.363259,2.163264e-03
25,N-metab,Ya,"poly(ZT, 3):phys",1.436262e-02,0.345650,3.347173e-02
